# Multi-Core Distributed PyTorch Training with TorchTPU on GKE

This notebook demonstrates how to execute **Distributed Data Parallel (DDP)** training across all attached TPU cores on a single-host Cloud TPU slice using **TorchTPU (`torch_tpu`)** and **`torchrun`**.

> [!IMPORTANT]
> **TPU Hardware Lock Isolation**:
> Cloud TPU character devices (`/dev/vfio/*`) are claimed exclusively by whichever process first initializes PJRT or touches `torch.device("tpu")`.
> In an interactive development notebook (like `torch_tpu_training.ipynb`), the interactive Python kernel maintains this hardware lock for single-core experiments.
> **This notebook is designed as an orchestrator**: its kernel does **not** import or initialize TPU devices directly. This leaves the TPU hardware completely unlocked so that child worker processes launched via `torchrun` can acquire the devices without resource contention!

## 1. Hardware Discovery, Pre-Flight Lock Verification & Inferring WORLD_SIZE

We verify that TPU hardware is completely free and unheld by another session, check that this orchestrator kernel has not claimed the devices, and infer `WORLD_SIZE` from `/dev/vfio`.

> [!IMPORTANT]
> **Orchestrator Kernel Isolation**: Child worker processes launched by `torchrun` require exclusive access to the `/dev/vfio/*` character devices. This notebook's kernel must **not** initialize `torch.device("tpu")` directly. The pre-flight check validates both external locks and current kernel state.

In [ ]:
import math
import os
import pathlib
import re
import sys

# Ensure tpu_lock utility is available
for p in [pathlib.Path.cwd(), pathlib.Path.cwd() / "examples" / "torch_tpu", pathlib.Path.cwd().parent]:
    if (p / "tpu_lock.py").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

import tpu_lock

# 1. Pre-flight check: Verify hardware is not locked to another session and current kernel is clean
ready = tpu_lock.preflight_check(is_distributed=True, auto_clear=True)
if not ready:
    raise RuntimeError("TPU hardware lock check failed. Please resolve the lock before proceeding.")

# 2. Infer local WORLD_SIZE from /dev/vfio count or TPU_TOPOLOGY env var
def get_local_world_size() -> int:
    """Infers local WORLD_SIZE from /dev/vfio count or TPU_TOPOLOGY env var."""
    devices = tpu_lock.get_tpu_devices()
    if devices:
        return len(devices)

    topology = os.environ.get("TPU_TOPOLOGY") or os.environ.get("TORCH_TPU_TOPOLOGY")
    if topology:
        try:
            dims = [int(p) for p in re.split(r"[,xX]", topology) if p]
            if dims:
                return math.prod(dims[:3])
        except Exception:
            pass

    return 1

world_size = get_local_world_size()
print(f"✅ Inferred local WORLD_SIZE = {world_size} TPU core(s)")

## 2. Standalone DDP Worker Script (`train_worker.py`)

Scaling from a single TPU core to multiple TPU cores uses PyTorch's native `DistributedDataParallel` (DDP) with the `tpu_dist` backend.

### Golden Rules of Distributed TPU:
1. **The `tpu_dist` Backend**: Initialized via `dist.init_process_group(backend="tpu_dist")`.
2. **Device Assignment**: On TPU, each worker process only "sees" its assigned core as `tpu`. Do **not** pass `device_ids` to `DDP(model)`.
3. **No Divergence**: Avoid `if rank == 0` guards around tensor operations or `.item()`. All ranks must execute identical execution graphs to prevent collective deadlocks.
4. **Shared Seeds**: Initialize weights with a shared seed across all ranks (`torch.manual_seed(42)`).

In [ ]:
%%writefile train_worker.py
# Standalone DDP training worker for TorchTPU
import os
import time
import torch
import torch.distributed as dist
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, TensorDataset, DistributedSampler
import numpy as np

class MLPClassifier(nn.Module):
    def __init__(self, in_features=784, hidden1=256, hidden2=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_features, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

def generate_synthetic_data(num_samples=8192, num_features=784, num_classes=10, seed=42):
    np.random.seed(seed)
    prototypes = np.random.randn(num_classes, num_features).astype(np.float32)
    labels = np.random.randint(0, num_classes, size=(num_samples,))
    noise = np.random.randn(num_samples, num_features).astype(np.float32) * 0.5
    images = prototypes[labels] + noise
    return torch.tensor(images, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)

def main():
    # 1. Initialize process group with tpu_dist backend before binding to TPU device
    if not dist.is_initialized():
        dist.init_process_group(backend="tpu_dist")
    device = torch.device("tpu")

    rank = dist.get_rank()
    world_size = dist.get_world_size()
    print(f"[Rank {rank}/{world_size}] Initialized TorchTPU DDP worker on device: {device}", flush=True)

    # 2. Set seed for deterministic initialization across ranks
    torch.manual_seed(42)

    # 3. Model setup on TPU wrapped in DDP (Note: no device_ids on TPU)
    model = MLPClassifier().to(device=device, dtype=torch.bfloat16)
    ddp_model = DDP(model)

    # 4. Distributed DataLoader
    images, labels = generate_synthetic_data(num_samples=8192, seed=42)
    dataset = TensorDataset(images, labels)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=64, sampler=sampler)

    loss_fn = nn.CrossEntropyLoss()
    optimizer = optim.SGD(ddp_model.parameters(), lr=0.05)

    epochs = 5
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        sampler.set_epoch(epoch)
        epoch_start = time.time()
        running_loss = 0.0
        total_samples = 0

        for batch_images, batch_labels in loader:
            batch_images = batch_images.to(device=device, dtype=torch.bfloat16)
            batch_labels = batch_labels.to(device=device)

            optimizer.zero_grad()
            outputs = ddp_model(batch_images)
            loss = loss_fn(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            # All ranks execute .item() synchronously to avoid divergence
            current_loss = loss.detach().item()
            running_loss += current_loss * batch_images.size(0)
            total_samples += batch_images.size(0)

        epoch_loss = running_loss / total_samples
        duration = time.time() - epoch_start
        throughput = (total_samples * world_size) / duration

        if rank == 0:
            print(f"Epoch {epoch:2d}/{epochs:2d} | Avg Loss: {epoch_loss:.4f} | Throughput: {throughput:.0f} samples/s ({duration:.2f}s)", flush=True)

    if rank == 0:
        print(f"\n[Rank 0] Distributed training complete in {time.time() - start_time:.2f}s across {world_size} TPU cores!", flush=True)

    dist.destroy_process_group()

if __name__ == "__main__":
    main()

## 3. Launching Multi-Core DDP with `torchrun`

We use TorchTPU's `set_tpu_launch_env` to bootstrap slicebuilder ports and topology, and launch `torchrun` across the attached TPU cores.
Because this orchestrator notebook did not initialize TPU hardware in its own kernel process, `/dev/vfio/*` is completely free for `torchrun`!

In [ ]:
import os
import socket
import subprocess
import sys
import tpu_lock
from torch.tpu.distributed import environment

environment.set_tpu_launch_env(nproc_per_node=world_size)

print(f"TORCH_TPU_TOPOLOGY: {os.environ.get('TORCH_TPU_TOPOLOGY')}")
print(f"TORCH_TPU_SLICEBUILDER_ADDRESSES: {os.environ.get('TORCH_TPU_SLICEBUILDER_ADDRESSES')}")

# Run distributed training locally via torchrun and stream output in real time
cmd = ["torchrun", f"--nproc_per_node={world_size}", "train_worker.py"]
print(f"Executing: {' '.join(cmd)}\n", flush=True)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    if process.returncode != 0:
        print(f"\n[ERROR] torchrun exited with return code {process.returncode}")
except KeyboardInterrupt:
    print("\n⚠️ Interrupted by user! Terminating child torchrun worker processes...", flush=True)
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    # Ensure any detached child workers are cleaned up
    tpu_lock.clear_tpu_locks(verbose=False)
    raise
finally:
    if process.poll() is None:
        process.terminate()

## 4. Hardware Verification & Post-Run Cleanup

When `torchrun` and its child worker processes terminate, their file descriptors to `/dev/vfio/*` are automatically released by the operating system.

The cell below verifies that no background worker processes or locks linger on the TPU devices, ensuring the hardware is immediately available for subsequent runs or for [`torch_tpu_training.ipynb`](torch_tpu_training.ipynb).

In [ ]:
# Verify that all TPU devices are completely free and no worker processes linger
import tpu_lock

print("🔍 Verifying TPU hardware lock status after distributed run...")
tpu_lock.check_tpu_locks(verbose=True)